#### reflection (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< sm'_operators

## reflection

In [ ]:
:= get_union_fields \* union_type. () : list (string * union_type) =
    ~>
        real_core::union_type_to_record
            `union_type
            \* union_record_type. =>
                real_core::record_type_fold
                    \ acc key =>
                        \* value. =>
                            := value =
                                ?: value ?|
                                | () => $'' : value
                                | _ =>
                                    backend_switch `value `({}) {
                                        Gleam =
                                            (\ () =>
                                                $'Nil' : value
                                            ) : () -> value
                                        Lua =
                                            (\ () =>
                                                $'nil' : value
                                            ) : () -> value
                                        Fsharp =
                                            (\ () =>
                                                $'Unchecked.defaultof<_>' : value
                                            ) : () -> value
                                        Python =
                                            (\ () =>
                                                := is_python =
                                                    backend_switch `bool `({}) {
                                                        Gleam = (\ () => !-) : () -> bool
                                                        Lua = (\ () => !-) : () -> bool
                                                        Fsharp = (\ () => !-) : () -> bool
                                                        Python = (\ () => !+) : () -> bool
                                                    }
                                                ? is_python ?>
                                                    ?: value ?|
                                                    | i8 => 0i8
                                                    | i16 => 0i16
                                                    | i32 => 0i32
                                                    | i64 => 0i64
                                                    | u8 => 0u8
                                                    | u16 => 0u16
                                                    | u32 => 0u32
                                                    | u64 => 0u64
                                                    | f32 => 0.0f32
                                                    | f64 => 0.0f64
                                                    | bool => !-
                                                    | char => '\0'
                                                    | _ => $'None' : value
                                                |: $'None' : value
                                            ) : () -> value
                                        Rust =
                                            (\ () =>
                                                $'Default::default()' : value
                                            ) : () -> value
                                        Zig =
                                            (\ () =>
                                                $'std.mem.zeroes(`value)' : value
                                            ) : () -> value
                                        TypeScript =
                                            (\ () =>
                                                ?: value ?|
                                                | i8 => 0i8
                                                | i16 => 0i16
                                                | i32 => 0i32
                                                | i64 => 0i64
                                                | u8 => 0u8
                                                | u16 => 0u16
                                                | u32 => 0u32
                                                | u64 => 0u64
                                                | f32 => 0.0f32
                                                | f64 => 0.0f64
                                                | bool => !-
                                                | char => '\0'
                                                | _ => $'undefined' : value
                                            ) : () -> value
                                    }
                            := item = real_core::nominal_create `union_type (key, value)
                            := key' = sm'_real::symbol_to_string_real `(`key)
                            (::) `(string * union_type) (key', item) acc
                    (Nil `(string * union_type))
                    `union_record_type
:| reflection_test_target = Browser | Contract

///- --timeout 900000

#### get_union_fields (test)

In [1]:
///- --test --symbol
///> rust
///> typescript
///> cuda

get_union_fields ()
|> listm'::box
|> listm'::to_array'
|> array
|> am'::sort_by snd
|> \ (array x : array _) => x
|> _assert_eq' (am'::base ;[ "Browser", Browser; "Contract", Contract ])
get_union_fields ()
|> listm'::box
|> listm'::to_array'
|> array
|> am'::sort_by snd
|> \ (array x : array _) => x
|> _assert_eq' (am'::base ;[ "Some", Some 0i32; "None", None ])

.rs (rust)
__assert_eq' { expected = [(Browser, Browser), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }
.ts (typescript)
__assert_eq' { expected = [(Browser, Browser), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }
.py (cuda)
__assert_eq' { expected = [(Browser, Browser), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }

0

In [ ]:
:= get_union_fields_untag \* union_type. () : list (string * union_type) =
    ~>
        real_core::union_type_to_record
            `union_type
            \* union_record_type. =>
                := result =
                    real_core::record_type_fold_back
                        \ _key =>
                            \* value. (acc, (i : i32)) =>
                                := key, item : (string * union_type) =
                                    real_core::union_untag `union_type i
                                        (\ key => \* value. =>
                                            := key' = sm'_real::symbol_to_string_real `(`key)
                                            := value =
                                                ?: value ?|
                                                | () => $'' : value
                                                | _ =>
                                                    backend_switch `value `({}) {
                                                        Gleam =
                                                            (\ () =>
                                                                $'Nil' : value
                                                            ) : () -> value
                                                        Lua =
                                                            (\ () =>
                                                                $'nil' : value
                                                            ) : () -> value
                                                        Fsharp =
                                                            (\ () =>
                                                                $'Unchecked.defaultof<_>' : value
                                                            ) : () -> value
                                                        Python =
                                                            (\ () =>
                                                                $'None' : value
                                                            ) : () -> value
                                                        Rust =
                                                            (\ () =>
                                                                $'Default::default()' : value
                                                            ) : () -> value
                                                        TypeScript =
                                                            (\ () =>
                                                                $'undefined' : value
                                                            ) : () -> value
                                                        Zig =
                                                            (\ () =>
                                                                $'std.mem.zeroes(`value)' : value
                                                            ) : () -> value
                                                    }
                                            := item = real_core::nominal_create `union_type (key, value)
                                            key', item
                                        )
                                        (\ _ =>
                                            failwith
                                                `(string * union_type)
                                                "reflection.get_union_fields_untag / invalid tag"
                                        )
                                (::) `(string * union_type) (key, item) acc, (+) `i32 i 1
                        `union_record_type
                        (Nil `(string * union_type), 0i32)
                := result = fst `(list (string * union_type)) `i32 result
                listm::rev `(string * union_type) result

///- --timeout 900000

#### get_union_fields_untag (test)

In [2]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

get_union_fields_untag ()
|> _assert_eq' [ "Browser", Browser; "Contract", Contract ]

get_union_fields_untag ()
|> _assert_eq' [ "Some", Some (); "None", None ]

.gleam (gleam)
__assert_eq' { expected = Cons((Browser, Browser), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.lua (lua)
__assert_eq' { expected = Cons((Browser, Browser), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.py (cuda)
__assert_eq' { expected = Cons((Browser, Browser), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.rs (rust)
__assert_eq' { expected = Cons((Browser, Browser), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.ts (typescript)
__assert_eq' { expected = Cons((Browser, Browser), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }

0

In [ ]:
:= union_try_pick \* t. (key : string) : option t =
    := fields = ~> get_union_fields_untag `t ()
    fields
    ++ (
        fields
        |> listm::map \ key', x =>
            key' |> sm'::to_lower, x
    )
    |> listm'::box
    |> listm'::to_array'
    |> \ x => array x : array _
    |> am'::try_pick \ key', x =>
        ? key' =. key
        ?> Some x
        |: None

///- --timeout 900000

#### union_try_pick (test)

In [3]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

"Browser"
|> union_try_pick
|> _assert_eq (Some Browser)

"Browser"
|> union_try_pick
|> _assert_eq (Some Browser)

.gleam (gleam)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.lua (lua)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.py (cuda)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.rs (rust)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.ts (typescript)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }

0

In [ ]:
:= union_try_pick' \* t. (key : string) : option t =
    := fields = ~> get_union_fields_untag `t ()
    fields
    ++ (
        fields
        |> listm::map \ key', x =>
            key' |> sm'::to_lower, x
    )
    |> listm'::try_pick \ key', x =>
        ? key' =. key
        ?> x |> Some
        |: None

///- --timeout 900000

In [4]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

"Browser"
|> union_try_pick'
|> _assert_eq (Some Browser)

"Browser"
|> union_try_pick'
|> _assert_eq (Some Browser)

.gleam (gleam)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.lua (lua)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.py (cuda)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.rs (rust)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }
.ts (typescript)
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = Some(Browser) }

0

In [ ]:
:= union_cases_fold \* union_type state. (step : state -> union_type -> state) (initial : state) : state =
    ~>
        real_core::union_type_to_record `union_type \* union_record_type. =>
            real_core::record_type_fold
                \ acc key => \* value. => step acc (real_core::nominal_create `union_type (key, ()))
                initial
                `union_record_type

///- --timeout 900000

#### union_cases_fold (test)

In [5]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

union_cases_fold (\ count (_ : reflection_test_target) => count + 1i32) 0i32
|> _assert_eq 2i32

union_cases_fold (\ found (target : reflection_test_target) => ? target = Contract ?> !+ |: found) !-
|> _assert_eq !+

.gleam (gleam)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.lua (lua)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = 2 }
__assert_eq { expected = true }

0

In [ ]:
:= union_cases_find \* union_type. (matches : union_type -> bool) : option union_type =
    :~ ^ search (cases : list union_type) : option union_type =
        ?= cases ?|
        | case :: rest => ? matches case ?> Some case |: search rest
        | [] => None
    search (union_cases_fold (\ acc (case : union_type) => case :: acc) [] |> dyn)

///- --timeout 900000

#### union_cases_find (test)

In [6]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

union_cases_find \ (target : reflection_test_target) => target = Contract
|> _assert_eq (Some Contract)

union_cases_find \ (_ : reflection_test_target) => !-
|> _assert_eq None

.gleam (gleam)
__assert_eq { expected = Some(Contract) }
__assert_eq { expected = None }
.lua (lua)
__assert_eq { expected = Some(Contract) }
__assert_eq { expected = None }
.py (cuda)
__assert_eq { expected = Some(Contract) }
__assert_eq { expected = None }
.rs (rust)
__assert_eq { expected = Some(Contract) }
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = Some(Contract) }
__assert_eq { expected = None }

0

In [ ]:
:= union_to_string \* t. (x : t) : string =
    ~> get_union_fields_untag `t ()
    |> listm'::try_pick \ key, x' =>
        ? x' = x
        ?> Some key
        |:
            := has_case =
                ~>
                    real_core::union_type_to_record
                        `t
                        \* union_record_type. =>
                            real_core::record_type_fold_back
                                \ _key =>
                                    \* value. acc =>
                                        ? acc
                                        ?> acc
                                        |:
                                            ?: value ?|
                                            | () => !-
                                            | _ => !+
                                `union_record_type
                                !-
            ? has_case |> not
            ?> None
            |:
                := x' = #?x' |> sm'::split "(" |> am'::index_base 0

                ? #?x |> sm'::starts_with x'
                ?> Some key
                |: None
    |> optionm::value

///- --timeout 900000

#### union_to_string (test)

In [7]:
///- --test --symbol
///> cuda
///> rust
///> typescript

Some !+
|> union_to_string
|> _assert_eq' "Some"

.py (cuda)
__assert_eq' { expected = Some }
.rs (rust)
__assert_eq' { expected = Some }
.ts (typescript)
__assert_eq' { expected = Some }

0

### case_name

In [ ]:
:= case_name \* t. (x : t) : string = !!!!UnionCaseName(x)

### case_name_styles

In [ ]:
:= case_name_lower \* t. (x : t) : string = !!!!UnionCaseName(x, "lower")
:= case_name_snake \* t. (x : t) : string = !!!!UnionCaseName(x, "snake")
:= case_name_kebab \* t. (x : t) : string = !!!!UnionCaseName(x, "kebab")
:= case_initial \* t. (x : t) : string = !!!!UnionCaseName(x, "initial")

### case_of_name

In [ ]:
:= case_of_name \* t. (text : string) : option t =
    ~> get_union_fields_untag `t ()
    |> listm'::try_pick \ key, case => ? key =. text ?> Some case |: None

### typed_name

In [ ]:
:! typed_name t = string

:= typed_case_name \* t. (x : t) : typed_name t = typed_name (case_name x)
:= case_qualified_name \* t. (x : t) : string = !!!!NominalName(`t) ++# "::" ++# case_name x

:= typed_name_text \* t. ((typed_name text) : typed_name t) : string = text

:= case_of_typed_name \* t. ((typed_name text) : typed_name t) : option t = case_of_name text

:= styled_step (separator : string) (acc : list string) (c : char) : list string =
    := s = sm'::of_char c
    := lower = sm'::to_lower s
    ? lower <> s && listm::length acc > 0i32 ?> lower :: separator :: acc |: lower :: acc

:= styled_name (separator : string) (text : string) : string =
    sm'::to_char_list text |> listm::fold (styled_step separator) [] |> listm::rev |> sm'::concat_list ""

:= typed_name_snake \* t. ((typed_name text) : typed_name t) : string = styled_name "_" text

:= typed_name_kebab \* t. ((typed_name text) : typed_name t) : string = styled_name "-" text

:= typed_name_lower \* t. ((typed_name text) : typed_name t) : string = sm'::to_lower text

///- --timeout 900000

#### case_name (test)

In [8]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

Contract
|> case_name
|> _assert_eq' "Contract"

Some 3i32
|> case_name
|> _assert_eq' "Some"

:= ~payload : option i32 = None
payload
|> case_name
|> _assert_eq' "None"

"Browser"
|> case_of_name
|> _assert_eq (Some Browser)

"browser"
|> case_of_name
|> _assert_eq (None : option reflection_test_target)

.gleam (gleam)
__assert_eq' { expected = Contract }
__assert_eq' { expected = Some }
__assert_eq' { expected = None }
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = None }
.lua (lua)
__assert_eq' { expected = Contract }
__assert_eq' { expected = Some }
__assert_eq' { expected = None }
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = None }
.py (cuda)
__assert_eq' { expected = Contract }
__assert_eq' { expected = Some }
__assert_eq' { expected = None }
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = None }
.rs (rust)
__assert_eq' { expected = Contract }
__assert_eq' { expected = Some }
__assert_eq' { expected = None }
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = None }
.ts (typescript)
__assert_eq' { expected = Contract }
__assert_eq' { expected = Some }
__assert_eq' { expected = None }
__assert_eq { expected = Some(Browser) }
__assert_eq { expected = None }

0

In [ ]:
:= nameof \* t. (x : t) : string =
    ~>
        real_core::record_type_fold_back
            \ key =>
                \* value. _ =>
                    sm'_real::symbol_to_string_real `(`key)
            `t
            ""

#### nameof (test)

In [9]:
///- --test --symbol

{ test1 = ""; test2 = "" }
|> nameof
|> _assert_eq' "test1"

__assert_eq' { expected = test1 }

In [ ]:
:= get_record_fields \* t u. (x : t) : list (string * u) =
    ~>
        real_core::record_type_fold_back
            \ key =>
                \* u'. acc =>
                    := k = sm'_real::symbol_to_string_real `(`key)
                    := v = x key
                    (::) `(string * u') (k, v) acc
            `t
            (Nil `(string * u))

#### get_record_fields (test)

In [10]:
///- --test --symbol

{ a = "1"; b = "2" }
|> get_record_fields
|> _assert_eq' [ "a", "1"; "b", "2" ]

__assert_eq' { expected = Cons((a, 1), Cons((b, 2), Nil)) }

In [ ]:
:= get_functions_types \* t {record}. (fns : t) =
    ~>
        := get_function_type \* t. =
            := args \* t {record}. : list (string * string) =
                real_core::record_type_fold_back
                    \ key =>
                        \* v. acc =>
                            := k = sm'_real::symbol_to_string_real `(`key)
                            := v = $'"`v"' : string
                            (::) `(string * string) (k, v) acc
                    `t
                    (Nil `(string * string))

            ?: t ?|
            | ~t -> ~u => args `t, ($'"`u"' : string)

        real_core::record_type_fold_back
            \ key =>
                \* v. acc =>
                    := k = sm'_real::symbol_to_string_real `(`key)
                    := args, result = get_function_type `v
                    (::) `(string * (list (string * string) * string)) (k, (args, result)) acc
            `(`fns)
            (Nil `(string * (list (string * string) * string)))
    |> \ x => x : list (string * (list (string * string) * string))

#### get_functions_types (test)

In [11]:
///- --test --no-agree "type names are backend-specific" --symbol
///> fsharp
///> rust

:= one ({ a } : { a : i32 }) : i32 = a + 1
:= two ({ a b } : { a : i32; b : i32 }) : i32 = a + b + 2
:= fns = { one two }
:= i32_name = run_target \| | Fsharp _ => \ () => ~~ "int32" | _ => \ () => "i32"

fns
|> get_functions_types
|> listm::map \ (name, args, result) => name, (args |> listm'::box |> listm'::to_array', result)
|> listm'::box
|> listm'::to_array'
|> sm'::format
|> _assert_eq' (
    [
        "one", [ "a", i32_name ], i32_name
        "two", [ "a", i32_name; "b", i32_name ], i32_name
    ]
    |> listm::map \ (name, args, result) => name, (args |> listm'::box |> listm'::to_array', result)
    |> listm'::box
    |> listm'::to_array'
    |> sm'::format
)

.fsx (fsharp)
__assert_eq' { expected = [(one, [(a, int32)], int32), (two, [(a, int32), (b, int32)], int32)] }
.rs (rust)
__assert_eq' { expected = [(one, [(a, i32)], i32), (two, [(a, i32), (b, i32)], i32)] }

0

In [ ]:
:| nameof_probe =
    | Alpha
    | Beta
    | Gamma
    | Delta
    | Epsilon

///- --timeout 900000

#### nameof_cases_round_trip (test)

In [12]:
///- --test --symbol
///> rust
///> gleam
///> lua
///> typescript
///> fsharp

Alpha |> case_name |> case_of_name |> _assert_eq (Some Alpha)
Beta |> case_name |> case_of_name |> _assert_eq (Some Beta)
Gamma |> case_name |> case_of_name |> _assert_eq (Some Gamma)
Delta |> case_name |> _assert_eq' "Delta"
Epsilon |> case_name |> case_of_name |> _assert_eq (Some Epsilon)
"epsilon" |> case_of_name |> _assert_eq (None : option nameof_probe)

.rs (rust)
__assert_eq { expected = Some(Alpha) }
__assert_eq { expected = Some(Beta) }
__assert_eq { expected = Some(Gamma) }
__assert_eq' { expected = Delta }
__assert_eq { expected = Some(Epsilon) }
__assert_eq { expected = None }
.gleam (gleam)
__assert_eq { expected = Some(Alpha) }
__assert_eq { expected = Some(Beta) }
__assert_eq { expected = Some(Gamma) }
__assert_eq' { expected = Delta }
__assert_eq { expected = Some(Epsilon) }
__assert_eq { expected = None }
.lua (lua)
__assert_eq { expected = Some(Alpha) }
__assert_eq { expected = Some(Beta) }
__assert_eq { expected = Some(Gamma) }
__assert_eq' { expected = Delta }
__assert_eq { expected = Some(Epsilon) }
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = Some(Alpha) }
__assert_eq { expected = Some(Beta) }
__assert_eq { expected = Some(Gamma) }
__assert_eq' { expected = Delta }
__assert_eq { expected = Some(Epsilon) }
__assert_eq { expected = None }
.fsx (fsharp)
__assert_eq { expected = Some(Alpha) }
_

0

In [ ]:
///- --timeout 900000

#### typed_case_name_round_trips (test)

In [13]:
///- --test --symbol
///> gleam
///> lua
///> rust
///> typescript

:= name = typed_case_name Contract
typed_name_text name |> _assert_eq' "Contract"
case_of_typed_name name |> _assert_eq (Some Contract)

.gleam (gleam)
__assert_eq' { expected = Contract }
__assert_eq { expected = Some(Contract) }
.lua (lua)
__assert_eq' { expected = Contract }
__assert_eq { expected = Some(Contract) }
.rs (rust)
__assert_eq' { expected = Contract }
__assert_eq { expected = Some(Contract) }
.ts (typescript)
__assert_eq' { expected = Contract }
__assert_eq { expected = Some(Contract) }

0

In [ ]:
///- --timeout 900000

#### typed_name_styles (test)

In [14]:
///- --test --symbol
///> gleam
///> rust

styled_name "_" "HostMacro" |> _assert_eq' "host_macro"
styled_name "-" "GadtCase" |> _assert_eq' "gadt-case"
styled_name "_" "Contract" |> _assert_eq' "contract"
:= name = typed_case_name Browser
typed_name_kebab name |> _assert_eq' "browser"
typed_name_lower name |> _assert_eq' "browser"

.gleam (gleam)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = contract }
__assert_eq' { expected = browser }
__assert_eq' { expected = browser }
.rs (rust)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = contract }
__assert_eq' { expected = browser }
__assert_eq' { expected = browser }

0

In [15]:
:| case_style_probe =
    | HostMacro
    | GadtCase

///- --timeout 900000

#### case_name_styles (test)

In [16]:
///- --test --symbol
///> gleam
///> lua
///> rust
///> typescript

:= ~macro : case_style_probe = HostMacro
macro |> case_name_snake |> _assert_eq' "host_macro"
macro |> case_name_lower |> _assert_eq' "hostmacro"
GadtCase |> case_name_kebab |> _assert_eq' "gadt-case"
GadtCase |> case_name_snake |> _assert_eq' "gadt_case"
macro |> case_initial |> _assert_eq' "h"
GadtCase |> case_initial |> _assert_eq' "g"

.gleam (gleam)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = hostmacro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = gadt_case }
__assert_eq' { expected = h }
__assert_eq' { expected = g }
.lua (lua)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = hostmacro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = gadt_case }
__assert_eq' { expected = h }
__assert_eq' { expected = g }
.rs (rust)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = hostmacro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = gadt_case }
__assert_eq' { expected = h }
__assert_eq' { expected = g }
.ts (typescript)
__assert_eq' { expected = host_macro }
__assert_eq' { expected = hostmacro }
__assert_eq' { expected = gadt-case }
__assert_eq' { expected = gadt_case }
__assert_eq' { expected = h }
__assert_eq' { expected = g }

0

In [ ]:
///- --timeout 900000

#### case_qualified_name (test)

In [17]:
///- --test --symbol
///> gleam
///> rust

case_qualified_name Contract |> _assert_eq' "reflection_test_target::Contract"
case_qualified_name Browser |> _assert_eq' "reflection_test_target::Browser"

.gleam (gleam)
__assert_eq' { expected = reflection_test_target::Contract }
__assert_eq' { expected = reflection_test_target::Browser }
.rs (rust)
__assert_eq' { expected = reflection_test_target::Contract }
__assert_eq' { expected = reflection_test_target::Browser }

0